### Step 0 Load Data & Inspect Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

df  = pd.read_csv("datasets/train.csv")

df.head()
print(df.info())
print(df.describe())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

#### Step 0.5: Check and manage null values

After studying the data to further understand the Meaning behind the columns with null values, it became further apparent that many of the null values were categorical so I can take the approach of encoding and just filling `Null` with `Missing`, and with the case of the numeric columns I came to the conclusion that they coulld be filled in with the median of the column to keep it even


In [2]:
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)
print(missing_percent)


PoolQC          99.520548
MiscFeature     96.301370
Alley           93.767123
Fence           80.753425
MasVnrType      59.726027
FireplaceQu     47.260274
LotFrontage     17.739726
GarageType       5.547945
GarageYrBlt      5.547945
GarageFinish     5.547945
GarageQual       5.547945
GarageCond       5.547945
BsmtExposure     2.602740
BsmtFinType2     2.602740
BsmtQual         2.534247
BsmtCond         2.534247
BsmtFinType1     2.534247
MasVnrArea       0.547945
Electrical       0.068493
dtype: float64


In [3]:
# Get only columns that have nulls
cols_with_nulls = df.columns[df.isnull().any()]

for col in cols_with_nulls:
    print(f"{col}: {df[col].dropna()}")

LotFrontage: 0       65.0
1       80.0
2       68.0
3       60.0
4       84.0
        ... 
1455    62.0
1456    85.0
1457    66.0
1458    68.0
1459    75.0
Name: LotFrontage, Length: 1201, dtype: float64
Alley: 21      Grvl
30      Pave
56      Pave
79      Grvl
87      Pave
        ... 
1404    Grvl
1414    Pave
1427    Grvl
1432    Grvl
1454    Pave
Name: Alley, Length: 91, dtype: object
MasVnrType: 0       BrkFace
2       BrkFace
4       BrkFace
6         Stone
7         Stone
         ...   
1446    BrkFace
1447    BrkFace
1451      Stone
1452    BrkFace
1456      Stone
Name: MasVnrType, Length: 588, dtype: object
MasVnrArea: 0       196.0
1         0.0
2       162.0
3         0.0
4       350.0
        ...  
1455      0.0
1456    119.0
1457      0.0
1458      0.0
1459      0.0
Name: MasVnrArea, Length: 1452, dtype: float64
BsmtQual: 0       Gd
1       Gd
2       Gd
3       TA
4       Gd
        ..
1455    Gd
1456    Gd
1457    TA
1458    TA
1459    TA
Name: BsmtQual, Length: 1423, 

#### Step 1: Separate target and features
* Log-transformed thethe target because SalePrice is right skewed.
* X will now contain all features (numeric + categorical).

In [4]:
y = np.log1p(df['SalePrice']) #Log to reduce skew
X = df.drop(['SalePrice'], axis=1)

#### Step 2: Split train/test first
* Split first to avoid data leakage when computing medians or encoders.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Step 3: Handlle numeric nulls
* Compute the median for training data 
* Apply same median to test data.

In [7]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    if X_train[col].isnull().sum() > 0:
        median = X_train[col].median()
        X_train[col].fillna(median, inplace=True)
        X_test[col].fillna(median, inplace=True)

#### Step 4: Handle categoricall nulls
* Each missing value becomes its own category
* Preserves row and encodes "absence" as informative


In [9]:
categorical_cols = X_train.select_dtypes(include=['object']).columns

for col in categorical_cols:
    if X_train[col].isnull().sum() > 0:
        X_train[col].fillna('Missing', inplace=True)
        X_test[col].fillna('Missing', inplace=True)

#### Step 5: One-hot encode categorical variables
* Aligning ensures both train and test matrices have same features.
* If test has new categories, they become 0 in train features (safe).

`drop_first`:
* `drop_first=True`: Removes the first encoded column to avoid multicollinearity (one column is linearly dependent on others). For a category with 3 values, you get 2 binary columns, not 3.


`reindex`:
* `reindex()`: Reorders test columns to match train and fills missing ones with 0, ensuring same feature dimensionality.


In [10]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns in case train/test differ
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

#### Step 6: Train baseline Linear Regression
* Metrics give a baselline performance.


In [15]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Baseline R²:", r2_score(y_test, y_pred_lr))
print("Baseline RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))

Baseline R²: -17.71127475814189
Baseline RMSE: 1.8686169967058106


### Notes
#### 1A: Combine & Transform features
* We Want to combine features that make sense to be together:
    * e.g. `TotalLivingArea` = `GrLivArea` + `TotalBsmtSF`
* Transform skewed features with log
    * Ususally applied when a variable is right-skewed
        * e,g. SalePrice has a many normal priced homes then there are a few examples of **Extreme** prices that creates a **long right tail**
    * Linear regression assumes thaat:
        * Errors are roughly **normally distributed**
        * Relationships are somewhat linear
        * Variance is relatively constant (homoscedasticity: signifies a constant spread of data points around the regression line, ensuring reliable, stable model predictions and valid statistical inferences)
    * What taking the log does:
        * It compresses large valllues more than small vallues

        | Original  | log(value) |
        | --------- | ---------- |
        | 100,000   | 11.51      |
        | 200,000   | 12.20      |
        | 1,000,000 | 13.81      |
        * Basically shrinks large outliers, and pullls the right tail a bit more inward, making the distribution more symmetric
    * Imprtance for linear regression:
        * Makes the relationship more linear
        * Reduces Heteroscedasticity
        * Makes Coefficients interpretable as % Change
    * Big Picture 
        * This is feature engineering,
        * I'm not changing the model I am changing the geometry of the dataspace
        * Often improves performance more than switching algorithms
    * `log1p()` vs `log()`:
        * `log1p`:
            * log(1 + x)
            * Prevents errors if there are zeros.
    * When should I use log:
        * Data is heavily right-skewed
        * Target spans large ranges
        * Residuals grow with magnitude
        * Relationship looks exponential
    * Summary of log
        * It allows for a more accuarate and linear regression.

* `get_dummies()`: performs one-hot encoding
    * Converts categorical variables to numerical binary columns so ml models can use them.

* `.fit()`:
    * Train the model using the data/ learn something from the data
    * Computes the `Mean and Standard deviation` of each column
* `.transform()`:
    * Apply what you learned from the transformations
* `.fit_transform()`:
    * A shortuct coombining `.fit()` and `.transform()`

# Part 2
#### Step 0: Scal numeric features for regularization & Set up polynomial features (Optional)
* Scaling:
    * Required for `Ridge/Lasso`
* Polynomial Features:
    * Adds interaction and squred terms
    * Can improve fit, but could also cause overfitting


In [12]:
# Scaling numeric vals
scaler= StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Poly features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

#### Step 1: Import Ridge and calculate the Cross-Validation (RMSE) `manual CV`
This step computes **5-fold CV RMSE(Root Mean Square Error)** to get a more reliable estimate than a single train/split.
* `alpha` controllls regularization strength: higher -> more shrinkage
* Ridge (L2) shriks coefficinets but keeps all feastures
* Lasso (L1) can zero out less important features, performing feature selcection.

In [13]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score

ridge = Ridge(alpha=1, random_state=42) # L2 regullalrization
lasso = Lasso(alpha=0.01, max_iter=10000, random_state=42)

def cv_rmse(model, X, y, cv=5):
    scores = cross_val_score(
        model, X, y, scoring='neg_root_mean_squared_error', cv=cv
    )
    return -scores.mean() # negative because sklearn returns negative RMSE

#Evaluate Ridge
ridge_rmse = cv_rmse(ridge, X_train_scaled, y_train)
lasso_rmse = cv_rmse(lasso, X_train_scaled, y_train)
print("Ridge CV RMSE:", ridge_rmse)
print("Lasso CV RMSE:", lasso_rmse)

Ridge CV RMSE: 0.1842437633215441
Lasso CV RMSE: 0.16408705755912015


#### Step 1.5: Automatic CV for Ridge and Lasso


In [14]:
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Initialize models
lasso_cv = LassoCV(alphas=[0.001, 0.01, 0.1, 1], cv=5, max_iter=10000, random_state=42)
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 13), cv=5)  # max_iter & random_state removed

# Fit models
lasso_cv.fit(X_train_scaled, y_train)
ridge_cv.fit(X_train_scaled, y_train)

# Predict on test set
y_pred_lassoCV = lasso_cv.predict(X_test_scaled)
y_pred_ridgeCV = ridge_cv.predict(X_test_scaled)

# Evaluate LassoCV
lassoCV_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lassoCV))
lassoCV_r2 = r2_score(y_test, y_pred_lassoCV)
print(f"LassoCV RMSE: {lassoCV_rmse:.4f}, R²: {lassoCV_r2:.4f}")
print("Chosen LassoCV Alpha:", lasso_cv.alpha_)

# Evaluate RidgeCV
ridgeCV_rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridgeCV))
ridgeCV_r2 = r2_score(y_test, y_pred_ridgeCV)
print(f"RidgeCV RMSE: {ridgeCV_rmse:.4f}, R²: {ridgeCV_r2:.4f}")
print("Chosen RidgeCV Alpha:", ridge_cv.alpha_)

LassoCV RMSE: 0.1471, R²: 0.8840
Chosen LassoCV Alpha: 0.01
RidgeCV RMSE: 0.1640, R²: 0.8558
Chosen RidgeCV Alpha: 316.22776601683796


##### Quick takeaway from step 1 & 1.5:
Why LassoCV / Ridge CV are better
1. Automatic hyperparameter tuning
    * They pick the best alpha via cross-validation taking out the guess work
    * Reduces risk of overfitting or underfitting compared to using a fixed alpha.
2. More reliablle generalization
    * Because they use CV on the training setm the sellected alpha tends to perform better on unseen data.
3. Cleaner workflow
    * One estimator both training and hyperparameter selection.
    * Less boilerplate and manually doing CV + Lass/Ridge + Grid search.
4. Interpertability (especially LassoCv)
    * LassoCV can zero out unimportant features automatically resulllting in it being easier to see which features matter.

### Step 2: Compare Against Baseline Linear Regression
This step is used to confirm whether regularization actually helped.

We compare:
| Model             | RMSE | R² |
| ----------------- | ---- | -- |
| Linear Regression | ?    | ?  |
| RidgeCV           | ?    | ?  |
| LassoCV           | ?    | ?  |

Whichever has:
* Lowest RMSE
* Highest R²

→ Generalizes best

In [50]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2 = r2_score(y_test, y_pred_lr)
print(f"|Model            | RMSE   | R²       |")
print(f"|-----------------|-------------------|")
print(f"|Linear Regression| {lr_rmse:.4f} |  {lr_r2:.4f}|")
print(f"|RidgeCV          | {ridgeCV_rmse:.4f} |  {ridgeCV_r2:.4f}  |")
print(f"|LassoCV          | {lassoCV_rmse:.4f} |  {lassoCV_r2:.4f}  |")


|Model            | RMSE   | R²       |
|-----------------|-------------------|
|Linear Regression| 1.8686 |  -17.7113|
|RidgeCV          | 0.1640 |  0.8558  |
|LassoCV          | 0.1471 |  0.8840  |


#### Step 3: Feature Importance Analysis
Taking a closer look at what LassoCV did

This will show:
* Which features matter most
* Which are eliminated

In [40]:
lasso_coeffs = pd.Series(lasso_cv.coef_, index=X_train.columns)

# Non-zero coefficients
important_features = lasso_coeffs[lasso_coeffs != 0].sort_values(key=abs, ascending=False)
print("Top 15 Important Features:")
print(important_features.head(15))

Top 15 Important Features:
GrLivArea               0.115401
OverallQual             0.106112
GarageCars              0.048087
YearBuilt               0.038737
OverallCond             0.029543
PoolQC_Gd              -0.023215
YearRemodAdd            0.023037
BsmtFullBath            0.022736
MSZoning_RM            -0.021871
TotalBsmtSF             0.017910
Condition2_PosN        -0.017206
CentralAir_Y            0.017009
Neighborhood_NridgHt    0.013712
Foundation_PConc        0.012986
FireplaceQu_Missing    -0.012909
dtype: float64


#### Step 6: Polynomial Features (Controllled Complexity)
Now we introduce feature expansion **carefully**, because this is where overfitting begins.

First retrain RidgeCV onlly (Ridge handles polynomial explosion better than Llasso).

In [53]:
ridge_poly = RidgeCV(alphas=np.logspace(-3, 3, 13), cv=5)
ridge_poly.fit(X_train_poly, y_train)

y_pred_poly = ridge_poly.predict(X_test_poly)

poly_rmse = np.sqrt(mean_squared_error(y_test, y_pred_poly))
poly_r2 = r2_score(y_test, y_pred_poly)

print(f"Polynomial RidgeCV RMSE: {poly_rmse:.4f}, R²: {poly_r2:.4f}")


Polynomial RidgeCV RMSE: 0.2799, R²: 0.5801
|Model            | RMSE   | R²       |
|-----------------|-------------------|
|RidgeCV          | 0.1640 |  0.8558  |
|LassoCV          | 0.1471 |  0.8840  |
|Poly RidgeCV     | 0.2799 |  0.5801  |


In [54]:
print(f"|Model            | RMSE   | R²       |")
print(f"|-----------------|-------------------|")
print(f"|RidgeCV          | {ridgeCV_rmse:.4f} |  {ridgeCV_r2:.4f}  |")
print(f"|LassoCV          | {lassoCV_rmse:.4f} |  {lassoCV_r2:.4f}  |")
print(f"|Poly RidgeCV     | {poly_rmse:.4f} |  {poly_r2:.4f}  |")

|Model            | RMSE   | R²       |
|-----------------|-------------------|
|RidgeCV          | 0.1640 |  0.8558  |
|LassoCV          | 0.1471 |  0.8840  |
|Poly RidgeCV     | 0.2799 |  0.5801  |


#### Quick Takeaways:
Why Polynomial Made It Worse:

I already had:
* One-hot encoded categorical variables
* Many feautres already

When I applied: 
    
    `PolynomialFeatures(degree=2)`

I created:
* Every squared term
* Every pairwise interaction

This massively increased my **variance**, **noise** and most importantly for the **Risk of Overfitting**
---
What the Results mean:

My dataset likely:
* Has strong linear signal
* Already benefits from one-hot encoding
* does not need poly expansion

So I shoulld conclude:
1. Poylnomial features are unnecessary here.
2. Regularization alone captures enough structure.
3. LassoCV is currently my best model.

Polynomial features are often useful when:
* I have mostly continuous numeric features
* Feature count is small
* I suspect nonlinear curvature

They are usually not ideal when:
* Feature space is already high-dimensional due to one-hot encoding.